In [32]:
import numpy as np
import os
import pandas as pd
import yaml, shutil
from ultralytics import YOLO
import subprocess, importlib
import subprocess, time, os


 ### Verifying Input Sources

In [3]:
print("=== Scanning /kaggle/input/ ===\n")
for folder in os.listdir("/kaggle/input"):
    full_path = f"/kaggle/input/{folder}"
    contents = os.listdir(full_path)
    print(f"📁 {full_path}")
    print(f"   {contents[:6]}\n")

=== Scanning /kaggle/input/ ===

📁 /kaggle/input/datasets
   ['henningheyen', 'hugodarwood']



 ### Adding LVIS Dataset and Epicurious Recipe Dataset

In [4]:
def explore(path, depth=0, max_depth=3):
    if depth > max_depth:
        return
    for item in os.listdir(path):
        full = os.path.join(path, item)
        print("  " * depth + f"{'📁' if os.path.isdir(full) else '📄'} {item}")
        if os.path.isdir(full) and depth < max_depth:
            explore(full, depth+1, max_depth)

print("=== LVIS Dataset ===")
explore("/kaggle/input/datasets/henningheyen")

print("\n=== Epicurious Dataset ===")
explore("/kaggle/input/datasets/hugodarwood")

=== LVIS Dataset ===
📁 lvis-fruits-and-vegetables-dataset
  📁 LVIS_Fruits_And_Vegetables
    📁 labels
      📁 val
      📁 test
      📁 train
    📄 data.yaml
    📁 images
      📁 val
      📁 test
      📁 train

=== Epicurious Dataset ===
📁 epirecipes
  📄 recipe.py
  📄 utils.py
  📄 full_format_recipes.json
  📄 epi_r.csv


In [ ]:
import importlib

def is_installed(pkg):
    try:
        importlib.import_module(pkg)
        return True
    except ImportError:
        return False

if not is_installed("ultralytics"):
    import subprocess
    subprocess.run(["pip", "install", "ultralytics", "-q"], check=True)
    print("✅ Ultralytics installed")
else:
    print("✅ Ultralytics already installed")

import ultralytics
print(f"   Version: {ultralytics.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 112.3 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

✅ Ultralytics installed
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
   Version: 8.4.73


### Defining All Paths

In [ ]:

LVIS_ROOT    = "/kaggle/input/datasets/henningheyen/lvis-fruits-and-vegetables-dataset/LVIS_Fruits_And_Vegetables"
DATA_YAML    = os.path.join(LVIS_ROOT, "data.yaml")
RECIPE_CSV   = "/kaggle/input/datasets/hugodarwood/epirecipes/epi_r.csv"

OUTPUT_DIR   = "/kaggle/working/recipe_generator"
WEIGHTS_DIR  = os.path.join(OUTPUT_DIR, "weights")
RUNS_DIR     = os.path.join(OUTPUT_DIR, "runs")

os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(RUNS_DIR,    exist_ok=True)

paths = {"data.yaml": DATA_YAML, "Recipe CSV": RECIPE_CSV, "LVIS root": LVIS_ROOT}
for name, path in paths.items():
    status = "" if os.path.exists(path) else ""
    print(f"{status} {name}: {path}")

✅ data.yaml: /kaggle/input/datasets/henningheyen/lvis-fruits-and-vegetables-dataset/LVIS_Fruits_And_Vegetables/data.yaml
✅ Recipe CSV: /kaggle/input/datasets/hugodarwood/epirecipes/epi_r.csv
✅ LVIS root: /kaggle/input/datasets/henningheyen/lvis-fruits-and-vegetables-dataset/LVIS_Fruits_And_Vegetables


### Inspecting data.yaml

In [7]:
with open(DATA_YAML, "r") as f:
    print(f.read())

# Train/val/test sets as 1) dir: path/to/imgs, 2) file: path/to/imgs.txt, or 3) list: [path/to/imgs1, path/to/imgs2, ..]
path: LVIS_Fruits_And_Vegetables # dataset root dir
train: images/train # 1000 train images (relative to 'path') 
val: images/test # 7221 val images (relative to 'path') 
test: images/test # 180 Manually labeled test images 

names:
  0: almond
  1: apple
  2: apricot
  3: artichoke
  4: asparagus
  5: avocado
  6: banana
  7: bean curd/tofu
  8: bell pepper/capsicum
  9: blackberry
  10: blueberry
  11: broccoli
  12: brussels sprouts
  13: cantaloup/cantaloupe
  14: carrot
  15: cauliflower
  16: cayenne/cayenne spice/cayenne pepper/cayenne pepper spice/red pepper/red pepper
  17: celery
  18: cherry
  19: chickpea/garbanzo
  20: chili/chili vegetable/chili pepper/chili pepper vegetable/chilli/chilli vegetable/chilly/chilly
  21: clementine
  22: coconut/cocoanut
  23: edible corn/corn/maize
  24: cucumber/cuke
  25: date/date fruit
  26: eggplant/aubergine
  27: f

### Inspecting Recipe CSV

In [8]:
df = pd.read_csv(RECIPE_CSV)
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSample row:")
print(df.iloc[0])

Shape: (20052, 680)

Columns: ['title', 'rating', 'calories', 'protein', 'fat', 'sodium', '#cakeweek', '#wasteless', '22-minute meals', '3-ingredient recipes', '30 days of groceries', 'advance prep required', 'alabama', 'alaska', 'alcoholic', 'almond', 'amaretto', 'anchovy', 'anise', 'anniversary', 'anthony bourdain', 'aperitif', 'appetizer', 'apple', 'apple juice', 'apricot', 'arizona', 'artichoke', 'arugula', 'asian pear', 'asparagus', 'aspen', 'atlanta', 'australia', 'avocado', 'back to school', 'backyard bbq', 'bacon', 'bake', 'banana', 'barley', 'basil', 'bass', 'bastille day', 'bean', 'beef', 'beef rib', 'beef shank', 'beef tenderloin', 'beer', 'beet', 'bell pepper', 'berry', 'beverly hills', 'birthday', 'biscuit', 'bitters', 'blackberry', 'blender', 'blue cheese', 'blueberry', 'boil', 'bok choy', 'bon appétit', 'bon app��tit', 'boston', 'bourbon', 'braise', 'bran', 'brandy', 'bread', 'breadcrumbs', 'breakfast', 'brie', 'brine', 'brisket', 'broccoli', 'broccoli rabe', 'broil', 'b

### Fixing data.yaml

In [ ]:
with open(DATA_YAML, "r") as f:
    config = yaml.safe_load(f)
config["path"]  = LVIS_ROOT
config["train"] = "images/train"
config["val"]   = "images/val"
config["test"]  = "images/test"
FIXED_YAML = "/kaggle/working/data.yaml"
with open(FIXED_YAML, "w") as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print("✅ Fixed data.yaml saved to:", FIXED_YAML)
print("\nContents:")
with open(FIXED_YAML) as f:
    print(f.read())

✅ Fixed data.yaml saved to: /kaggle/working/data.yaml

Contents:
names:
  0: almond
  1: apple
  2: apricot
  3: artichoke
  4: asparagus
  5: avocado
  6: banana
  7: bean curd/tofu
  8: bell pepper/capsicum
  9: blackberry
  10: blueberry
  11: broccoli
  12: brussels sprouts
  13: cantaloup/cantaloupe
  14: carrot
  15: cauliflower
  16: cayenne/cayenne spice/cayenne pepper/cayenne pepper spice/red pepper/red pepper
  17: celery
  18: cherry
  19: chickpea/garbanzo
  20: chili/chili vegetable/chili pepper/chili pepper vegetable/chilli/chilli vegetable/chilly/chilly
  21: clementine
  22: coconut/cocoanut
  23: edible corn/corn/maize
  24: cucumber/cuke
  25: date/date fruit
  26: eggplant/aubergine
  27: fig/fig fruit
  28: garlic/ail
  29: ginger/gingerroot
  30: Strawberry
  31: gourd
  32: grape
  33: green bean
  34: green onion/spring onion/scallion
  35: Tomato
  36: kiwi fruit
  37: lemon
  38: lettuce
  39: lime
  40: mandarin orange
  41: melon
  42: mushroom
  43: onion
  

### Session-Safe Training

In [ ]:
TRAIN_OUTPUT = "/kaggle/working/recipe_generator/runs/train"
BEST_WEIGHTS = os.path.join(TRAIN_OUTPUT, "weights/best.pt")
LAST_WEIGHTS = os.path.join(TRAIN_OUTPUT, "weights/last.pt")

# If training already completed, skip
if os.path.exists(BEST_WEIGHTS):
    print("Training already complete. Loading existing weights:")
    print(f"   {BEST_WEIGHTS}")
    model = YOLO(BEST_WEIGHTS)

# If training was interrupted, resume
elif os.path.exists(LAST_WEIGHTS):
    print("Incomplete training found. Resuming...")
    model = YOLO(LAST_WEIGHTS)
    model.train(resume=True)

# Fresh training
else:
    print("Starting fresh training...")
    model = YOLO("yolov8n.pt")  
    model.train(
        data    = FIXED_YAML,
        epochs  = 50,
        imgsz   = 640,
        batch   = 16,
        name    = "train",
        project = "/kaggle/working/recipe_generator/runs",
        exist_ok= True,
        cache   = True,       
        patience= 10,         
        device  = 0,          
        workers = 2,
    )
    print(f"\nTraining complete. Best weights at:\n   {BEST_WEIGHTS}")

Starting fresh training...
Ultralytics 8.4.73 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=T

### Verifying Weights Saved

In [13]:
if os.path.exists(BEST_WEIGHTS):
    size_mb = os.path.getsize(BEST_WEIGHTS) / (1024*1024)
    print(f"best.pt exists — {size_mb:.1f} MB")
else:
    print("best.pt not found — training may still be running or failed")

best.pt exists — 6.0 MB


### Check Training Results

In [ ]:
BEST_WEIGHTS = "/kaggle/working/recipe_generator/runs/train/weights/best.pt"
model = YOLO(BEST_WEIGHTS)
results_csv = "/kaggle/working/recipe_generator/runs/train/results.csv"
if os.path.exists(results_csv):
    import pandas as pd
    results = pd.read_csv(results_csv)
    results.columns = results.columns.str.strip()
    last = results.iloc[-1]
    print("=== Training Results (Final Epoch) ===")
    print(f"  Box Precision : {float(last['metrics/precision(B)']):.3f}")
    print(f"  Recall        : {float(last['metrics/recall(B)']):.3f}")
    print(f"  mAP@50        : {float(last['metrics/mAP50(B)']):.3f}")
    print(f"  mAP@50-95     : {float(last['metrics/mAP50-95(B)']):.3f}")
else:
    print("results.csv not found, but weights exist")
    print(f"   Model classes: {model.names}")

size_mb = os.path.getsize(BEST_WEIGHTS) / (1024*1024)
print(f"\nbest.pt size: {size_mb:.1f} MB")
print(f"   Total classes: {len(model.names)}")

=== Training Results (Final Epoch) ===
  Box Precision : 0.394
  Recall        : 0.151
  mAP@50        : 0.114
  mAP@50-95     : 0.074

best.pt size: 6.0 MB
   Total classes: 63


### Diagnosing Data Quality

In [17]:
splits = ["train", "val", "test"]
for split in splits:
    img_dir = f"{LVIS_ROOT}/images/{split}"
    lbl_dir = f"{LVIS_ROOT}/labels/{split}"
    
    imgs = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    lbls = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    
    print(f"{split:5s} → images: {imgs:5d} | labels: {lbls:5d} | match: {'✅' if imgs==lbls else '❌'}")

# Check a sample label file
import glob
sample_labels = glob.glob(f"{LVIS_ROOT}/labels/train/*.txt")[:3]
print(f"\nSample label files:")
for lbl in sample_labels:
    with open(lbl) as f:
        content = f.read().strip()
    print(f"  {os.path.basename(lbl)}: {content[:120]}")

train → images:     1 | labels:     1 | match: ✅
val   → images:     1 | labels:     1 | match: ✅
test  → images:   180 | labels:   180 | match: ✅

Sample label files:


### Quick Inference Test

In [ ]:
# Testing on a sample image to see if model detects anything at all
import glob

model = YOLO(BEST_WEIGHTS)
sample_imgs = glob.glob(f"{LVIS_ROOT}/images/test/*.jpg")[:5]

for img in sample_imgs:
    results = model(img, verbose=False)
    boxes = results[0].boxes
    if boxes is not None and len(boxes) > 0:
        detected = [model.names[int(c)] for c in boxes.cls]
        confs    = [round(float(s), 2) for s in boxes.conf]
        print(f"✅ {os.path.basename(img)}: {list(zip(detected, confs))}")
    else:
        print(f"❌ {os.path.basename(img)}: nothing detected")

✅ 20240402_3f803a6c-9204-4ddd-934a-cbbbefbdf14f_3_png.rf.7cf31e858e4f7e090520307c5cbfff3d.jpg: [('broccoli', 0.35)]
✅ 20240327_ad61f4d1-5de3-4bc0-9ca1-5739e6c27b93_2_png.rf.297c02e7f7151bf8246477ab71a5373d.jpg: [('banana', 0.77), ('banana', 0.42)]
✅ 20240329_6f2c8e4e-302b-4053-bc4f-87f96a6b3004_1_png.rf.70bfad0a286d20c8d8e77c98943bf482.jpg: [('orange/orange fruit', 0.94), ('orange/orange fruit', 0.92), ('orange/orange fruit', 0.91), ('orange/orange fruit', 0.9), ('orange/orange fruit', 0.89), ('orange/orange fruit', 0.89), ('orange/orange fruit', 0.88), ('orange/orange fruit', 0.86), ('orange/orange fruit', 0.83), ('orange/orange fruit', 0.82), ('orange/orange fruit', 0.78)]
❌ 20240327_3f803a6c-9204-4ddd-934a-cbbbefbdf14f_3_png.rf.2efd53a8c45735c9f54d1d3a42fa7f7a.jpg: nothing detected
✅ 20240328_3f803a6c-9204-4ddd-934a-cbbbefbdf14f_1_png.rf.da0521eb0f9ec766c068a3dc48c35c8c.jpg: [('tomato', 0.92), ('tomato', 0.9), ('tomato', 0.9), ('tomato', 0.9), ('tomato', 0.86), ('tomato', 0.84), ('t

### Switching to Base YOLOv8n

In [ ]:

# Using base pretrained model instead of our poorly-trained one
model = YOLO("yolov8n.pt")

# Checking what food-related classes it knows from COCO
food_related = {k: v for k, v in model.names.items() 
                if any(f in v.lower() for f in 
                ['apple','banana','orange','broccoli','carrot',
                 'hot dog','pizza','donut','cake','sandwich',
                 'fork','knife','spoon'])}

print("Food-related COCO classes available:")
for k, v in food_related.items():
    print(f"  {k}: {v}")

Food-related COCO classes available:
  42: fork
  43: knife
  44: spoon
  46: banana
  47: apple
  48: sandwich
  49: orange
  50: broccoli
  51: carrot
  52: hot dog
  53: pizza
  54: donut
  55: cake


### Testing Base Model on Same Images

In [19]:
sample_imgs = glob.glob(f"{LVIS_ROOT}/images/test/*.jpg")[:5]

for img in sample_imgs:
    results = model(img, verbose=False, conf=0.25)
    boxes = results[0].boxes
    if boxes is not None and len(boxes) > 0:
        detected = [model.names[int(c)] for c in boxes.cls]
        confs    = [round(float(s), 2) for s in boxes.conf]
        print(f"✅ {os.path.basename(img)[:40]}: {list(zip(detected, confs))}")
    else:
        print(f"❌ {os.path.basename(img)[:40]}: nothing detected")

✅ 20240402_3f803a6c-9204-4ddd-934a-cbbbefb: [('person', 0.88), ('dining table', 0.38), ('bowl', 0.32)]
✅ 20240327_ad61f4d1-5de3-4bc0-9ca1-5739e6c: [('banana', 0.82), ('dining table', 0.26)]
✅ 20240329_6f2c8e4e-302b-4053-bc4f-87f96a6: [('dining table', 0.54), ('bowl', 0.54), ('orange', 0.4), ('orange', 0.33), ('orange', 0.3), ('orange', 0.29), ('orange', 0.29), ('carrot', 0.26)]
✅ 20240327_3f803a6c-9204-4ddd-934a-cbbbefb: [('bottle', 0.81), ('person', 0.8)]
✅ 20240328_3f803a6c-9204-4ddd-934a-cbbbefb: [('fork', 0.91), ('knife', 0.84), ('dining table', 0.56), ('dining table', 0.51), ('carrot', 0.5), ('carrot', 0.49), ('carrot', 0.27)]


### Combing Both Models(fine-tuned best.pt + Base yolov8n.pt)

In [21]:

# Load both models
model_lvis = YOLO(BEST_WEIGHTS)   # our 63-class model
model_coco = YOLO("yolov8n.pt")   # base COCO model

# COCO food-only class ids we care about
COCO_FOOD_IDS = {46:'banana', 47:'apple', 48:'sandwich', 49:'orange',
                 50:'broccoli', 51:'carrot', 52:'hot dog', 53:'pizza',
                 54:'donut', 55:'cake'}

def detect_ingredients(image_path, conf_lvis=0.25, conf_coco=0.30):
    ingredients = set()
    
    # Model 1 — LVIS (our fine-tuned, 63 classes)
    r1 = model_lvis(image_path, verbose=False, conf=conf_lvis)
    if r1[0].boxes is not None:
        for cls, conf in zip(r1[0].boxes.cls, r1[0].boxes.conf):
            name = model_lvis.names[int(cls)]
            # Clean up long class names
            name = name.split("/")[0].strip()
            ingredients.add(name.lower())
    
    # Model 2 — COCO (only food classes)
    r2 = model_coco(image_path, verbose=False, conf=conf_coco)
    if r2[0].boxes is not None:
        for cls, conf in zip(r2[0].boxes.cls, r2[0].boxes.conf):
            cls_id = int(cls)
            if cls_id in COCO_FOOD_IDS:
                ingredients.add(COCO_FOOD_IDS[cls_id].lower())
    
    return sorted(ingredients)



### Quick Test

In [22]:
# Quick test
import glob
sample_imgs = glob.glob(f"{LVIS_ROOT}/images/test/*.jpg")[:5]
for img in sample_imgs:
    detected = detect_ingredients(img)
    print(f"📸 {os.path.basename(img)[:45]}")
    print(f"   🥕 Detected: {detected}\n")

📸 20240402_3f803a6c-9204-4ddd-934a-cbbbefbdf14f
   🥕 Detected: ['broccoli']

📸 20240327_ad61f4d1-5de3-4bc0-9ca1-5739e6c27b93
   🥕 Detected: ['banana']

📸 20240329_6f2c8e4e-302b-4053-bc4f-87f96a6b3004
   🥕 Detected: ['orange']

📸 20240327_3f803a6c-9204-4ddd-934a-cbbbefbdf14f
   🥕 Detected: []

📸 20240328_3f803a6c-9204-4ddd-934a-cbbbefbdf14f
   🥕 Detected: ['carrot', 'lettuce', 'tomato']



### Loading & Preparing Recipe Engine

In [ ]:
df = pd.read_csv(RECIPE_CSV)

meta_cols = ['rating', 'calories', 'protein', 'fat', 'sodium']
ingredient_cols = [c for c in df.columns if c not in meta_cols + ['title']]

df[ingredient_cols] = df[ingredient_cols].fillna(0)

print(f"Recipes loaded: {len(df)}")
print(f"Ingredient columns: {len(ingredient_cols)}")

# Check which of our detected classes exist as CSV columns
our_classes = [model_lvis.names[i].split('/')[0].strip().lower() 
               for i in model_lvis.names]
coco_food   = list(COCO_FOOD_IDS.values())
all_detected_classes = set(our_classes + coco_food)

matched = [c for c in all_detected_classes if c in ingredient_cols]
print(f"\nDetectable ingredients that exist in recipe CSV: {len(matched)}")
print(f"   {sorted(matched)}")

Recipes loaded: 20052
Ingredient columns: 674

Detectable ingredients that exist in recipe CSV: 52
   ['almond', 'apple', 'apricot', 'artichoke', 'asparagus', 'avocado', 'banana', 'bell pepper', 'blackberry', 'blueberry', 'broccoli', 'cake', 'carrot', 'cauliflower', 'celery', 'cherry', 'chickpea', 'chili', 'coconut', 'cucumber', 'date', 'eggplant', 'fig', 'garlic', 'ginger', 'grape', 'green bean', 'lemon', 'lettuce', 'lime', 'melon', 'mushroom', 'onion', 'orange', 'papaya', 'pea', 'peach', 'pear', 'persimmon', 'pineapple', 'pizza', 'potato', 'prune', 'pumpkin', 'radish', 'raspberry', 'sandwich', 'strawberry', 'tomato', 'turnip', 'watermelon', 'zucchini']


Recipe Matching Function

In [ ]:
def find_recipes(detected_ingredients, df, ingredient_cols, top_n=5):
    if not detected_ingredients:
        return pd.DataFrame()
    
    valid = [i for i in detected_ingredients if i in ingredient_cols]
    
    if not valid:
        return pd.DataFrame()
    
    scores = df[valid].sum(axis=1)           
    total  = df[ingredient_cols].sum(axis=1)
    bonus  = scores / total.replace(0, 1)
    
    combined = scores + (0.3 * bonus)
    
    df_scored = df[['title'] + valid].copy()
    df_scored['match_count'] = scores
    df_scored['score']       = combined.round(3)
    df_scored = df_scored[df_scored['match_count'] > 0]
    df_scored = df_scored.sort_values('score', ascending=False).head(top_n)
    
    return df_scored.reset_index(drop=True)


def get_recipe_details(title, df):
    """Get full ingredient list for a recipe title."""
    row = df[df['title'] == title].iloc[0]
    meta = ['title','rating','calories','protein','fat','sodium']
    ingredients_in_recipe = [
        col for col in df.columns 
        if col not in meta and row[col] == 1.0
    ]
    return {
        'title'      : title,
        'rating'     : row.get('rating', 'N/A'),
        'calories'   : row.get('calories', 'N/A'),
        'ingredients': ingredients_in_recipe
    }

### Testing Recipe Matching

In [ ]:
test_cases = [
    ['broccoli'],
    ['banana'],
    ['orange'],
    ['carrot', 'lettuce', 'tomato'],
    ['garlic', 'onion', 'tomato', 'mushroom'], 
]

for ingredients in test_cases:
    print(f"\n Detected: {ingredients}")
    print("-" * 50)
    matches = find_recipes(ingredients, df, ingredient_cols, top_n=3)
    if matches.empty:
        print(" No matches found (ingredients not in CSV columns)")
    else:
        for _, row in matches.iterrows():
            print(f"    {row['title']}")
            print(f"      Matched {int(row['match_count'])} ingredient(s) | score: {row['score']}")


 Detected: ['broccoli']
--------------------------------------------------
    Curried Stir-Fried Noodles with Vegetables 
      Matched 1 ingredient(s) | score: 1.05
    Orecchiette, Broccoli 
      Matched 1 ingredient(s) | score: 1.05
    Curried Stir-Fried Noodles with Vegetables 
      Matched 1 ingredient(s) | score: 1.05

 Detected: ['banana']
--------------------------------------------------
    Mocha Berry-Almond Smoothie 
      Matched 1 ingredient(s) | score: 1.06
    Peanut Butter and Banana Sandwiches 
      Matched 1 ingredient(s) | score: 1.06
    Banana Chocolate Tart with Caramel and Chocolate Sauces 
      Matched 1 ingredient(s) | score: 1.05

 Detected: ['orange']
--------------------------------------------------
    Orange-Cardamom Spritzer 
      Matched 1 ingredient(s) | score: 1.1
    Orange Dust 
      Matched 1 ingredient(s) | score: 1.075
    Orange Dust 
      Matched 1 ingredient(s) | score: 1.075

 Detected: ['carrot', 'lettuce', 'tomato']
-------------

### Full Pipeline Test (Detection → Recipe)

In [ ]:

sample_imgs = glob.glob(f"{LVIS_ROOT}/images/test/*.jpg")[:5]

for img_path in sample_imgs:
    print(f"\nImage: {os.path.basename(img_path)[:45]}")
    
    detected = detect_ingredients(img_path)
    print(f"  Detected: {detected}")
    
    matches = find_recipes(detected, df, ingredient_cols, top_n=3)
    
    if matches.empty:
        print("    No recipe matches found")
    else:
        print("   Top recipes:")
        for _, row in matches.iterrows():
            print(f"      → {row['title']} (matched: {int(row['match_count'])})")


Image: 20240402_3f803a6c-9204-4ddd-934a-cbbbefbdf14f
  Detected: ['broccoli']
   Top recipes:
      → Curried Stir-Fried Noodles with Vegetables  (matched: 1)
      → Orecchiette, Broccoli  (matched: 1)
      → Curried Stir-Fried Noodles with Vegetables  (matched: 1)

Image: 20240327_ad61f4d1-5de3-4bc0-9ca1-5739e6c27b93
  Detected: ['banana']
   Top recipes:
      → Mocha Berry-Almond Smoothie  (matched: 1)
      → Peanut Butter and Banana Sandwiches  (matched: 1)
      → Banana Chocolate Tart with Caramel and Chocolate Sauces  (matched: 1)

Image: 20240329_6f2c8e4e-302b-4053-bc4f-87f96a6b3004
  Detected: ['orange']
   Top recipes:
      → Orange-Cardamom Spritzer  (matched: 1)
      → Orange Dust  (matched: 1)
      → Orange Dust  (matched: 1)

Image: 20240327_3f803a6c-9204-4ddd-934a-cbbbefbdf14f
  Detected: []
    No recipe matches found

Image: 20240328_3f803a6c-9204-4ddd-934a-cbbbefbdf14f
  Detected: ['carrot', 'lettuce', 'tomato']
   Top recipes:
      → California Garden Roll  (

### Fixing Duplicates 

In [ ]:
def find_recipes(detected_ingredients, df, ingredient_cols, top_n=5):
    if not detected_ingredients:
        return pd.DataFrame()
    
    valid = [i for i in detected_ingredients if i in ingredient_cols]
    if not valid:
        return pd.DataFrame()
    
    scores   = df[valid].sum(axis=1)
    total    = df[ingredient_cols].sum(axis=1)
    bonus    = scores / total.replace(0, 1)
    combined = scores + (0.3 * bonus)
    
    df_scored = df[['title'] + valid].copy()
    df_scored['match_count'] = scores
    df_scored['score']       = combined.round(3)
    df_scored = df_scored[df_scored['match_count'] > 0]
    
    # Fix: drop duplicate titles, keep highest score
    df_scored = df_scored.sort_values('score', ascending=False)
    df_scored = df_scored.drop_duplicates(subset='title')
    
    return df_scored.head(top_n).reset_index(drop=True)

print("Recipe function updated")

Recipe function updated


### Installing Streamlit

In [30]:
if not importlib.util.find_spec("streamlit"):
    subprocess.run(["pip", "install", "streamlit", "pyngrok", "-q"], check=True)
    print("Streamlit + pyngrok installed")
else:
    print("Already installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 94.7 MB/s eta 0:00:00
Streamlit + pyngrok installed


Writing the Streamlit App

In [31]:
app_code = '''
import streamlit as st
from ultralytics import YOLO
import pandas as pd
import numpy as np
from PIL import Image
import tempfile, os

# ── Page config ──────────────────────────────────────────────
st.set_page_config(
    page_title="Recipe Generator",
    page_icon="🍽️",
    layout="centered"
)

# ── Load models & data (cached) ───────────────────────────────
@st.cache_resource
def load_models():
    m1 = YOLO("/kaggle/working/recipe_generator/runs/train/weights/best.pt")
    m2 = YOLO("yolov8n.pt")
    return m1, m2

@st.cache_data
def load_recipes():
    df = pd.read_csv("/kaggle/input/datasets/hugodarwood/epirecipes/epi_r.csv")
    meta = ["rating","calories","protein","fat","sodium","title"]
    ingredient_cols = [c for c in df.columns if c not in meta]
    df[ingredient_cols] = df[ingredient_cols].fillna(0)
    return df, ingredient_cols

model_lvis, model_coco = load_models()
df, ingredient_cols    = load_recipes()

COCO_FOOD_IDS = {
    46:"banana", 47:"apple", 48:"sandwich", 49:"orange",
    50:"broccoli", 51:"carrot", 52:"hot dog", 53:"pizza",
    54:"donut", 55:"cake"
}

# ── Core functions ────────────────────────────────────────────
def detect_ingredients(image_path, conf_lvis=0.25, conf_coco=0.30):
    ingredients = set()
    r1 = model_lvis(image_path, verbose=False, conf=conf_lvis)
    if r1[0].boxes is not None:
        for cls in r1[0].boxes.cls:
            name = model_lvis.names[int(cls)].split("/")[0].strip().lower()
            ingredients.add(name)
    r2 = model_coco(image_path, verbose=False, conf=conf_coco)
    if r2[0].boxes is not None:
        for cls in r2[0].boxes.cls:
            cid = int(cls)
            if cid in COCO_FOOD_IDS:
                ingredients.add(COCO_FOOD_IDS[cid])
    return sorted(ingredients)

def find_recipes(detected, top_n=5):
    if not detected:
        return pd.DataFrame()
    valid = [i for i in detected if i in ingredient_cols]
    if not valid:
        return pd.DataFrame()
    scores   = df[valid].sum(axis=1)
    total    = df[ingredient_cols].sum(axis=1)
    bonus    = scores / total.replace(0, 1)
    combined = scores + (0.3 * bonus)
    out = df[["title"] + valid].copy()
    out["match_count"] = scores.values
    out["score"]       = combined.round(3)
    out = out[out["match_count"] > 0]
    out = out.sort_values("score", ascending=False)
    out = out.drop_duplicates(subset="title")
    return out.head(top_n).reset_index(drop=True)

# ── UI ────────────────────────────────────────────────────────
st.title("🍽️ AI Recipe Generator")
st.markdown("Upload a photo of your ingredients and get instant recipe suggestions.")
st.divider()

uploaded = st.file_uploader(
    "📸 Upload an ingredient photo",
    type=["jpg","jpeg","png"]
)

conf_thresh = st.slider("Detection confidence", 0.10, 0.80, 0.25, 0.05)

if uploaded:
    # Save to temp file
    with tempfile.NamedTemporaryFile(delete=False, suffix=".jpg") as tmp:
        tmp.write(uploaded.read())
        tmp_path = tmp.name

    col1, col2 = st.columns(2)
    with col1:
        st.image(tmp_path, caption="Uploaded image", use_column_width=True)

    with st.spinner("🔍 Detecting ingredients..."):
        detected = detect_ingredients(tmp_path, conf_lvis=conf_thresh, conf_coco=conf_thresh)

    with col2:
        st.markdown("### 🥕 Detected Ingredients")
        if detected:
            for ing in detected:
                st.success(f" {ing}")
        else:
            st.warning("No ingredients detected. Try a clearer photo or lower the confidence slider.")

    st.divider()

    if detected:
        with st.spinner("🍳 Finding best recipes..."):
            matches = find_recipes(detected, top_n=5)

        if matches.empty:
            st.warning("Ingredients detected but no matching recipes found. Try different ingredients.")
        else:
            st.markdown("### 🍽️ Top Recipe Suggestions")
            for i, row in matches.iterrows():
                matched_cols = [c for c in row.index 
                                if c not in ["title","match_count","score"] 
                                and row[c] == 1]
                with st.expander(f"{'🥇' if i==0 else '🥈' if i==1 else '🥉' if i==2 else '▪️'} {row['title']}"):
                    st.markdown(f"**Matched ingredients:** {', '.join(matched_cols)}")
                    st.markdown(f"**Match count:** {int(row['match_count'])}  |  **Score:** {row['score']}")
                    recipe_row = df[df["title"] == row["title"]].iloc[0]
                    all_ings = [c for c in ingredient_cols if recipe_row[c] == 1.0]
                    st.markdown(f"**All recipe ingredients:** {', '.join(all_ings[:20])}")

    os.unlink(tmp_path)
'''

with open("/kaggle/working/app.py", "w") as f:
    f.write(app_code)

print("app.py saved to /kaggle/working/app.py")

app.py saved to /kaggle/working/app.py


### Launching the App

In [ ]:
from pyngrok import ngrok
import subprocess, time

NGROK_TOKEN = "3FSObMYOHlgUA6cL7sh3wtAu4sr_2enoPdb7zF54ZwoskeBi2"

subprocess.run(["ngrok", "config", "add-authtoken", NGROK_TOKEN], 
               capture_output=True)

time.sleep(2)

public_url = ngrok.connect(8501)
print(f"\n✅ App is live at: {public_url}")


✅ App is live at: NgrokTunnel: "https://embattled-uneatable-cupbearer.ngrok-free.dev" -> "http://localhost:8501"


# Conclusion

This project successfully demonstrates how computer vision and data-driven recommendation techniques can be combined to create an intelligent recipe generation system. By integrating a YOLO-based object detection model with a large recipe dataset, the application is able to identify ingredients from an uploaded image and recommend recipes that best match the detected items.

Throughout the development process, several practical challenges such as dataset preparation, model training, ingredient matching, and duplicate recommendations were addressed to improve the overall quality of the system. To enhance detection coverage, the solution also incorporates a pretrained YOLO model alongside the custom-trained model, resulting in more reliable ingredient recognition.

Although the system performs well for many common fruits and vegetables, its accuracy is still influenced by the quality of the training data and the range of supported ingredient classes. Future improvements could include training on larger and more diverse datasets, recognizing multiple objects with higher precision, incorporating quantity estimation, and generating personalized recipe suggestions based on dietary preferences or nutritional goals.

Overall, the project highlights the potential of artificial intelligence in simplifying everyday tasks such as meal planning and recipe discovery. It provides a practical example of how machine learning, computer vision, and data processing can be integrated into a user-friendly application that transforms visual input into meaningful recommendations.